In [ ]:
import pandas as pd

from pyfaidx import Fasta

from Bio import SeqIO

import matplotlib.pyplot as plt

import numpy as np

import seaborn as sns

from Bio.Seq import Seq


In [ ]:
                           

circids_df = pd.read_csv('circExor/resources/exoRase/circRNAs_anno.csv', usecols = ['circID', 'circBase ID', 'Genomic position', 'Strand'])

circids_df


In [ ]:
circids_df['circBase ID'].isnull().sum()


In [ ]:
                                 

EV_BRCA_df = pd.read_csv('circExor/resources/exoRase/EVPs_in_BRCA_circRNAs.txt', sep = '	')

EV_BRCA_df['sum_EVP'] = EV_BRCA_df.iloc[:, 1:].sum(axis=1)

EV_BRCA_df


In [ ]:
EV_BRCA_df['Genomic position'] = circids_df['Genomic position'].values

EV_BRCA_df['Strand'] = circids_df['Strand'].values

EV_BRCA_df


In [ ]:
EV_BRCA_df = EV_BRCA_df[['circID', 'sum_EVP', 'Genomic position', 'Strand']]

                    

EV_BRCA_df = EV_BRCA_df[EV_BRCA_df['sum_EVP'] != 0]

print(len(EV_BRCA_df))

print(EV_BRCA_df.head())


In [ ]:
                    

bed_df = pd.read_csv('circExor/resources/circAtlas/human_bed_v3.0.txt', sep = '	')

bed_df



In [ ]:
bed_df['circAltas_ID'].isnull().sum()


In [ ]:
                                            

EV_BRCA_df[['Chro', 'Pos']] = EV_BRCA_df['Genomic position'].str.split(':', expand=True)

EV_BRCA_df[['Start', 'End']] = EV_BRCA_df['Pos'].str.split('-', expand=True)

                

EV_BRCA_df['Start'] = EV_BRCA_df['Start'].astype(int)

EV_BRCA_df['End'] = EV_BRCA_df['End'].astype(int)

bed_df['Start'] = bed_df['Start'].astype(int)

bed_df['End'] = bed_df['End'].astype(int)

                                                       

matched_chunks = []

for d_start in [-1, 0, 1]:

    for d_end in [-1, 0, 1]:

        temp_ev = EV_BRCA_df.copy()

                               

        temp_ev['Start_match'] = temp_ev['Start'] + d_start

        temp_ev['End_match'] = temp_ev['End'] + d_end

        

                      

        chunk = pd.merge(

            temp_ev, 

            bed_df, 

            left_on=['Chro', 'Strand', 'Start_match', 'End_match'], 

            right_on=['Chro', 'Strand', 'Start', 'End'],

            suffixes=('_ev', '_bed')

        )

        matched_chunks.append(chunk)

                  

matched_df = pd.concat(matched_chunks, ignore_index=True)

matched_df = matched_df.drop_duplicates(subset=['circID', 'circAltas_ID'])

                                               

mapped_results = matched_df.groupby('circID')['circAltas_ID'].apply(lambda x: ','.join(x.unique())).reset_index()

                              

total_unique_circaltas = matched_df['circAltas_ID'].nunique()

total_matches = len(matched_df)

print(f"Total matched records: {total_matches} records.")

print(f"Found {total_unique_circaltas} unique circAltas_ID values.")

                           

mapped_circ_df = mapped_results.rename(columns={'circAltas_ID': 'matched_circAltas_IDs'})

                                                     

mapped_circ_df = mapped_circ_df.merge(EV_BRCA_df[['circID', 'sum_EVP']].drop_duplicates(), on='circID', how='left')

mapped_circ_df


In [ ]:
seq_df = pd.read_csv('circExor/resources/circAtlas/human_sequence_v3.0', sep = ' ', header=None, names=['circAltas_ID', 'Sequence'])

seq_df


In [ ]:
temp_df = mapped_circ_df.copy()

temp_df['single_ID'] = temp_df['matched_circAltas_IDs'].str.split(',')

temp_df = temp_df.explode('single_ID')

                           

merged_seq = pd.merge(temp_df, seq_df, left_on='single_ID', right_on='circAltas_ID', how='left')

                                           

seq_agg = merged_seq.groupby('circID')['Sequence'].apply(lambda x: ','.join(x.dropna().astype(str))).reset_index()

                                           

mapped_circ_df = pd.merge(mapped_circ_df, seq_agg, on='circID', how='left')

mapped_circ_df


In [ ]:
EV_BRCA_df = mapped_circ_df

               

original_row_count_ev = len(EV_BRCA_df)

            

                                   

                                 

                   

                    

EV_BRCA_df = EV_BRCA_df[

    (~EV_BRCA_df['Sequence'].str.contains('N', case=False, na=False)) & 

    (EV_BRCA_df['Sequence'] != "unknown") & 

    (EV_BRCA_df['Sequence'].notna()) &

    (EV_BRCA_df['Sequence'].str.len() > 0)

].copy()

                

EV_BRCA_df = EV_BRCA_df.reset_index(drop=True)

            

new_row_count_ev = len(EV_BRCA_df)

deleted_rows_ev = original_row_count_ev - new_row_count_ev

print("--- EV_BRCA_df cleaning report ---")

print(f"Original rows: {original_row_count_ev}")

print(f"Deleted rows: {deleted_rows_ev}")

print(f"Remaining rows: {new_row_count_ev}")


In [ ]:
        

EV_BRCA_df['Sequence_Length'] = EV_BRCA_df['Sequence'].str.len()

original_ev_count = len(EV_BRCA_df)

                           

EV_BRCA_df = EV_BRCA_df[EV_BRCA_df['Sequence_Length'] <= 11238].copy()

         

deleted_ev = original_ev_count - len(EV_BRCA_df)

print(f"EV_BRCA_df cleaning completed: deleted {deleted_ev} overlong sequences; remaining {len(EV_BRCA_df)} records.")


In [ ]:
            

max_length = EV_BRCA_df['Sequence_Length'].max()

min_length = EV_BRCA_df['Sequence_Length'].min()

average_length = EV_BRCA_df['Sequence_Length'].mean()

print(f"--- EV_BRCA_df length statistics ---")

print(f"Minimum sequence length: {min_length}")

print(f"Maximum sequence length: {max_length}")

print(f"Average sequence length: {average_length:.2f}")

                  

bins = np.histogram_bin_edges(EV_BRCA_df['Sequence_Length'], bins=30)

bin_counts, bin_edges = np.histogram(EV_BRCA_df['Sequence_Length'], bins=bins)

print("\nSequence length distribution by bins：")

for i in range(len(bin_counts)):

    print(f"{bin_edges[i]:.1f} - {bin_edges[i+1]:.1f}: {bin_counts[i]} sequences")

          

plt.figure(figsize=(10, 6))

sns.histplot(EV_BRCA_df['Sequence_Length'], bins=bins, kde=True, color='skyblue', label='Length Distribution')

                        

plt.axvline(min_length, color='green', linestyle='--', linewidth=1.5, label=f'Min: {min_length}')

plt.axvline(max_length, color='orange', linestyle='--', linewidth=1.5, label=f'Max: {max_length}')

plt.axvline(average_length, color='red', linestyle='--', linewidth=2, label=f'Mean: {average_length:.2f}')

           

plt.title('Sequence Length Distribution of EV_BRCA_df', fontsize=16)

plt.xlabel('Sequence Length (bp)', fontsize=12)

plt.ylabel('Frequency (Count)', fontsize=12)

plt.legend()

plt.grid(axis='y', alpha=0.3)

plt.tight_layout()

         

plt.show()


In [ ]:


stats = EV_BRCA_df['sum_EVP'].describe()

print(stats)


In [ ]:
EV_BRCA_df.to_csv('./exoRbase_output.bak.csv')
